<a href="https://colab.research.google.com/github/suchitapatil09/AI_LAB/blob/main/1BM24CS294_WEEK7_FOL_UNIFICATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
def is_variable(term):
    """Returns True if the term is a variable (lowercase string)."""
    return isinstance(term, str) and term.islower()

def occurs_in(var, expression):
    """Checks if a variable occurs inside another compound expression."""
    if isinstance(expression, str):
        return var == expression
    return any(occurs_in(var, term) for term in expression)

def apply_substitution(subst, expression):
    """Applies known substitutions to the remainder of expressions."""
    if isinstance(expression, str):
        return subst.get(expression, expression)
    return [apply_substitution(subst, term) for term in expression]

def unify(psi1, psi2):
    """Main unification algorithm."""
    # Step 1: Handle variables and constants
    if isinstance(psi1, str) or isinstance(psi2, str):
        if psi1 == psi2:
            return {}
        elif is_variable(psi1):
            if occurs_in(psi1, psi2):
                return "FAILURE"
            return {psi1: psi2}
        elif is_variable(psi2):
            if occurs_in(psi2, psi1):
                return "FAILURE"
            return {psi2: psi1}
        else:
            return "FAILURE"

    # Defensive check for compound expressions (lists)
    if not isinstance(psi1, list) or not isinstance(psi2, list):
        return "FAILURE"

    # Step 2: Check predicate symbols
    if psi1[0] != psi2[0]:
        return "FAILURE"

    # Step 3: Check number of arguments
    if len(psi1) != len(psi2):
        return "FAILURE"

    # Step 4: Initialize substitution set
    subst = {}

    # Step 5: Iterate through arguments
    for i in range(1, len(psi1)):
        # Apply any substitutions found so far to the current elements
        element1 = apply_substitution(subst, psi1[i])
        element2 = apply_substitution(subst, psi2[i])

        # Recursively unify the elements
        s = unify(element1, element2)

        if s == "FAILURE":
            return "FAILURE"

        if s != {}:
            subst.update(s)

    # Step 6: Return final substitution set
    return subst

def main():
    print("--- First-Order Logic Unification ---")

    # Test Case 1: From the presentation slides
    expr_A = ["Eats", "x", "Apple"]
    expr_B = ["Eats", "Riya", "y"]
    print(f"\nUnifying: {expr_A} and {expr_B}")
    print("Result:", unify(expr_A, expr_B))

    # Test Case 2: The dry-run example
    expr_C = ["Read", "x", "Book"]
    expr_D = ["Read", "Alex", "y"]
    print(f"\nUnifying: {expr_C} and {expr_D}")
    print("Result:", unify(expr_C, expr_D))

    # Test Case 3: Failing Unification (Predicate mismatch)
    expr_E = ["Knows", "John", "x"]
    expr_F = ["Eats", "John", "Apple"]
    print(f"\nUnifying: {expr_E} and {expr_F}")
    print("Result:", unify(expr_E, expr_F))

    # Test Case 4: Occurs Check Failure (Infinite Loop Prevention)
    expr_G = ["P", "x"]
    expr_H = ["P", ["f", "x"]]
    print(f"\nUnifying: {expr_G} and {expr_H}")
    print("Result:", unify(expr_G, expr_H))

if __name__ == "__main__":
    main()

--- First-Order Logic Unification ---

Unifying: ['Eats', 'x', 'Apple'] and ['Eats', 'Riya', 'y']
Result: {'x': 'Riya', 'y': 'Apple'}

Unifying: ['Read', 'x', 'Book'] and ['Read', 'Alex', 'y']
Result: {'x': 'Alex', 'y': 'Book'}

Unifying: ['Knows', 'John', 'x'] and ['Eats', 'John', 'Apple']
Result: FAILURE

Unifying: ['P', 'x'] and ['P', ['f', 'x']]
Result: FAILURE
